# Single Baseline Scaffolded and Feathered 1D DPSS Inpainting

**by Josh Dillon and Tyler Cox**, last updated September 24, 2026

This notebook performs single-baseline, full-night DPSS inpainting of the corner-turned, calibrated,
redundantly-averaged data under the night's final flags: the last step of the per-night pipeline,
whose products go to LST stacking. Where data are flagged, a *scaffold* (prior) set of visibilities
is substituted with feathered (distance-dependent sigmoid) weights, and a 1D DPSS model is fit
along the frequency axis, as in H6C's
[single_baseline_scaffolded_and_feathered_inpainter](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_scaffolded_and_feathered_inpainter.ipynb).
The scaffold is the LST-indexed single-baseline sky model of the same redundant group, matched to
the night's LSTs and rephased, when that baseline has one (autocorrelations included); otherwise it
is built from the night's own data by iteratively fitting a 2D DPSS model at increasing delays, as
in H6C's
[single_baseline_2D_informed_inpaint](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_2D_informed_inpaint.ipynb)
(a model that covers only part of the night informs that fit, and the model must not be flagged
where the data are not).
Either way, a 2D DPSS model of the night's autocorrelations provides the noise weights, and the
polarized point source models are subtracted first (from the scaffold too), so the inpainted data
stay source-subtracted. New here: at inpainted cells, the effective `nsamples` are filled in by a
low-delay DPSS fit across the gap, so that LST stacking weights the inpainted model by the depth the
night would have had there; the `where_inpainted` sidecar records which cells those are.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths and wrapper-level toggles.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: 4-Pol Phase and Amplitude Waterfalls Before and After Inpainting](#Figure-1:-4-Pol-Phase-and-Amplitude-Waterfalls-Before-and-After-Inpainting)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import yaml
import glob
import copy
import re
from astropy import units
from scipy import constants

from pyuvdata import UVFlag
from hera_cal import io, flag_utils, utils, polfilt, vis_clean, abscal, red_groups, datacontainer
from hera_cal.frf import sky_frates, get_FR_buffer_from_spectra
from hera_filters.dspec import dpss_operator, sparse_linear_fit_2D, fourier_filter
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline

## Parse inputs and outputs

To run interactively, provide a sum file path if the environment variable is not set; everything
else has defaults.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "SINGLE_BASELINE_SCAFFOLDED_INPAINT_NOTEBOOK")

# default names for the files this notebook reads or writes: single-baseline files end in the suffixes, per-night
# products are relative to SUM_FILE's folder ({JD} is the integer JD), and catalogs and scaffolds are absolute paths
SINGLE_BASELINE_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'
INPAINTED_SUFFIX = 'sum.inpainted.uvh5'
WHERE_INPAINTED_SUFFIX = 'sum.where_inpainted.h5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'
FLAGS_PI_FRF_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_pI_FRF.h5'  # the night's final flags
SOURCE_CATALOG_FILENAME = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/source_catalogs/sources.yaml'
SOURCE_MODELS_FILENAME = 'single_baseline_files/phased.{JD}.{source}.{model}.uvh5'
INPAINT_SCAFFOLD_FILENAME = '/lustre/aoc/projects/hera/h6c-analysis/abscal_models/h6c_filtered_lst_stack/single_baseline_files/zen.LST.baseline.{ant1}_{ant2}.sum.sky_model.uvh5'  # {ant1}_{ant2} is the redundant group's key; baselines without one use the 2D-informed fallback

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)

# default settings, overridden by the TOML's [SINGLE_BASELINE_SCAFFOLDED_INPAINT_OPTS] section (if given)
INPAINT_DELAY = 1000.0  # in ns; the 1D DPSS inpainting model's delay half-width (and the fallback 2D fit's final delay)
AUTO_INPAINT_DELAY = 100.0  # in ns; the smooth 2D model of the autocorrelations that sets the noise weights
NSAMPLES_FILL_DELAY = 100.0  # in ns; the fill of effective nsamples across inpainted gaps (nsamples structure is Tsys-shaped)
ITERATIVE_DELAY_DELTA = 25.0  # in ns; the fallback's 2D fits go from this delay up to INPAINT_DELAY in these steps
EIGENVAL_CUTOFF = 1e-12
CG_TOL = 1e-6  # conjugate-gradient tolerance of the 2D fits
CG_ITER_LIM = 500
INPAINT_WIDTH_FACTOR = 0.5  # feathering: flagged cells reach full weight this many (1 / INPAINT_DELAY) / df channels from unflagged data...
INPAINT_ZERO_DIST_WEIGHT = 1e-2  # ...starting from this relative weight right at a flag edge
AUTO_FR_SPECTRUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/beam_simulation_products/spectra_cache_hera_auto.h5'  # the autocorrelation's fringe-rate spectrum, for the 2D fits' time axis
GAUSS_FIT_BUFFER_CUT = 1e-5
SCAFFOLD_EXTRAP_LIMIT = 0.5  # in units of the model's LST spacing; a data time farther than this from any model LST has no scaffold
SUBTRACT_POLARIZED_SOURCE = True  # subtract per_night_source_filtering's Faraday-rotating (and scintillation) models before inpainting...
SUBTRACT_FG_MODEL = False  # ...and their smooth foreground models too
SUBTRACT_POLARIZED_SOURCE_FROM_SCAFFOLD = True  # the same models are subtracted from the sky-model scaffold
USE_LOW_BAND_MODEL = False  # subtract the source models below BAND_SPLIT_FREQ too
FILTER_SCAFFOLD = False  # low-pass delay filter a sky-model scaffold first (unnecessary for a 2D-DPSS-filtered model)
FILTER_STANDOFF = 50.0  # in ns; that filter's standoff beyond the baseline's horizon delay...
FILTER_MIN_DELAY = 300.0  # in ns; ...and its minimum half-width
FR0_FILTER = False  # also write a copy of the inpainted cross-correlations with a fringe-rate-zero notch applied
FR0_HALFWIDTH = 0.01  # in mHz

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'SINGLE_BASELINE_SCAFFOLDED_INPAINT_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

MODEL_COMPONENTS = (["fg_model"] if SUBTRACT_FG_MODEL else []) + ["rm_model", "scint_model"]
jdstr = re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)
CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
FINAL_FLAG_FILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_PI_FRF_FILENAME.format(JD=jdstr))

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SINGLE_BASELINE_SUFFIX', 'INPAINTED_SUFFIX', 'WHERE_INPAINTED_SUFFIX',
                'CORNER_TURN_MAP_YAML', 'FINAL_FLAG_FILE', 'SOURCE_CATALOG_FILENAME', 'SOURCE_MODELS_FILENAME', 'INPAINT_SCAFFOLD_FILENAME',
                'BAND_SPLIT_FREQ', 'INPAINT_DELAY', 'AUTO_INPAINT_DELAY', 'NSAMPLES_FILL_DELAY', 'ITERATIVE_DELAY_DELTA', 'EIGENVAL_CUTOFF',
                'CG_TOL', 'CG_ITER_LIM', 'INPAINT_WIDTH_FACTOR', 'INPAINT_ZERO_DIST_WEIGHT', 'AUTO_FR_SPECTRUM_FILE', 'GAUSS_FIT_BUFFER_CUT',
                'SCAFFOLD_EXTRAP_LIMIT', 'SUBTRACT_POLARIZED_SOURCE', 'SUBTRACT_FG_MODEL', 'SUBTRACT_POLARIZED_SOURCE_FROM_SCAFFOLD',
                'USE_LOW_BAND_MODEL', 'MODEL_COMPONENTS', 'FILTER_SCAFFOLD', 'FILTER_STANDOFF', 'FILTER_MIN_DELAY', 'FR0_FILTER', 'FR0_HALFWIDTH']:
    print(f'{setting} = {eval(setting)}')

In [ ]:
add_to_history = 'Produced by single_baseline_scaffolded_and_feathered_inpainter notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65

## Preliminaries

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

# the map is keyed by the night's redundantly averaged files; this job's share is under the one with SUM_FILE's JD
jd_str = re.search(r'zen\.(\d+\.\d+)\.', os.path.basename(SUM_FILE)).group(1)
single_bl_files = next((outfiles for red_avg_file, outfiles in corner_turn_map['files_to_outfiles_map'].items()
                        if f'zen.{jd_str}.' in os.path.basename(red_avg_file)), [])
print(f'The corner-turn map assigns {len(single_bl_files)} single-baseline files to zen.{jd_str}.')

# the night's final flags: the single-baseline files carry only the flags the calibration left
prior_flags = np.all(UVFlag(FINAL_FLAG_FILE).flag_array, axis=-1)
print(f'Final flags from {FINAL_FLAG_FILE}: {np.mean(prior_flags):.3%} flagged.')

In [ ]:
if SUBTRACT_POLARIZED_SOURCE:
    with open(SOURCE_CATALOG_FILENAME) as f:
        source_config = yaml.safe_load(f)

    model_specs = []
    for source in source_config["sources"]:
        fsname = source["name"].replace(" ", "_").replace("-", "_")
        for model_name in MODEL_COMPONENTS:
            model_file = os.path.join(os.path.dirname(SUM_FILE), SOURCE_MODELS_FILENAME.format(JD=jdstr, source=fsname, model=model_name))
            if os.path.isfile(model_file):
                model_specs.append((model_file, model_name))

    print(f"Using model components: {MODEL_COMPONENTS}")
    print(f"Found {len(model_specs)} model files" + (" (no polarized source was modeled on this night)." if len(model_specs) == 0 else ""))
    for model_file, model_name in model_specs:
        print(f"  {model_name}: {os.path.basename(model_file)}")

## Functions for main loop

In [ ]:
FR_CENTER_AND_HW_CACHE = {}

def cache_fr_center_and_hw(hd, antpair, tslice, band):
    '''Figure out the range of FRs in Hz spanned for a given band and tslice, buffered by the size of the autocorrelation FR kernel,
    and stores the value in FR_CENTER_AND_HW_CACHE (if it hasn't already been computed.'''
    if (tslice is not None) and (band is not None) and ((antpair, tslice, band) not in FR_CENTER_AND_HW_CACHE):
        # calculate fringe rate center and half-width and then update cache
        fr_buffer = get_FR_buffer_from_spectra(AUTO_FR_SPECTRUM_FILE, hd.times[tslice], hd.freqs[band], 
                                               gauss_fit_buffer_cut=GAUSS_FIT_BUFFER_CUT)
        hd_here = hd.select(inplace=False, frequencies=hd.freqs[band])
        fr_center = list(sky_frates(hd_here)[0].values())[0] / 1e3  # converts to Hz
        fr_hw = (list(sky_frates(hd_here)[1].values())[0] + fr_buffer) / 1e3    
        FR_CENTER_AND_HW_CACHE[(antpair, tslice, band)] = fr_center, fr_hw

In [ ]:
def get_ip_nsamples(nsamples, flags, tslices, bands):
    '''Effective nsamples at the flagged cells of every integration with some unflagged data in the band, for the inpainting
    weights and, at inpainted cells, for the output. Phase II's effective nsamples vary with frequency (they carry the
    autocorrelations' spectra) and are 0 wherever every member of the redundant group was flagged, so unlike H6C's per-band
    counts they must be filled in: by a DPSS fit (NSAMPLES_FILL_DELAY) along frequency of the integration's unflagged
    nsamples, clipped to their range. So LST stacking weights an inpainted model by the depth the night would have had
    there. Unflagged cells, and integrations fully flagged in the band (never inpainted, zero weight), are unchanged.'''
    ip_nsamples = copy.deepcopy(nsamples)
    for bl in nsamples:
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            rows = np.where(~np.all(flags[bl][tslice, band], axis=1))[0] + tslice.start
            if len(rows) == 0:
                continue
            wgts = np.where(flags[bl][rows][:, band], 0, 1).astype(float)
            fit, _, _ = fourier_filter(nsamples.freqs[band], nsamples[bl][rows][:, band].astype(float), wgts=wgts,
                                       filter_centers=[0], filter_half_widths=[NSAMPLES_FILL_DELAY * 1e-9], mode='dpss_solve',
                                       eigenval_cutoff=[EIGENVAL_CUTOFF], suppression_factors=[EIGENVAL_CUTOFF],
                                       max_contiguous_edge_flags=len(nsamples.freqs), filter_dims=1)
            for k, tind in enumerate(rows):
                unflagged = nsamples[bl][tind, band][wgts[k] > 0]
                fill = np.clip(fit[k], np.min(unflagged), np.max(unflagged))
                ip_nsamples[bl][tind, band] = np.where(flags[bl][tind, band], fill, nsamples[bl][tind, band])
    return ip_nsamples

In [ ]:
def get_weights_for_inpainting(data, flags, tslices, bands, ip_nsamples, ip_autos, auto_flags):
    '''Get inverse noise variance weights for inpainting. These come in two flavors:
        * weights_before_ip: has 0s wherever the data or autos are flagged
        * weights_after_ip: uses inpainted autos for "noise," so only has 0s wherever the
            autos weren't inpainted (in practice, nowhere in the bands/tslice of interest) and in
            integrations fully flagged in the band, which are never inpainted and whose (possibly
            zero) nsamples are never filled: nothing anchors the 2D fits there.
    '''
    weights_before_ip = {}
    weights_after_ip = {}
    for bl in data:
        ant1, ant2 = utils.split_bl(bl)
        
        auto_bl_1 = [k for k in ip_autos if k[2] == utils.join_pol(ant1[1], ant1[1])][0]
        auto_bl_2 = [k for k in ip_autos if k[2] == utils.join_pol(ant2[1], ant2[1])][0]
        with np.errstate(divide='ignore', invalid='ignore'):  # nsamples can be 0 in fully flagged integrations
            noise = (np.abs(ip_autos[auto_bl_1] * ip_autos[auto_bl_2]) / (ip_nsamples[bl] * dt * df))**.5

        # assign non-zero weights only in the bands/tslices of interest
        weights_before_ip[bl] = np.zeros_like(data[bl], dtype=float)
        weights_after_ip[bl] = np.zeros_like(data[bl], dtype=float)
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            
            non_finite_ip_auto = (~np.isfinite(ip_autos[auto_bl_1][tslice, band])) 
            non_finite_ip_auto |= (~np.isfinite(ip_autos[auto_bl_2][tslice, band]))
            fully_flagged = np.all(flags[bl][tslice, band], axis=1, keepdims=True)
            weights_after_ip[bl][tslice, band] = np.where(non_finite_ip_auto | fully_flagged, 0, noise[tslice, band]**-2)

            flags_here = auto_flags[auto_bl_1][tslice, band] | auto_flags[auto_bl_2][tslice, band] 
            flags_here |= flags[bl][tslice, band] | non_finite_ip_auto
            weights_before_ip[bl][tslice, band] = np.where(flags_here, 0, noise[tslice, band]**-2)

        # renormalize
        for wgts in [weights_after_ip[bl], weights_before_ip[bl]]:
            if np.any(wgts > 0):
                wgts /= np.mean(wgts[wgts > 0])
    return weights_before_ip, weights_after_ip

In [ ]:
def fit_2D_DPSS(data, weights, filter_delay, tslices, bands, **kwargs):
    '''Fit a 2D DPSS model to all the baselines in data. The time-dimension is based on sky FRs
    and the FR spectrum of the autos. fr_centers and fr_hws are drawn from FR_CENTER_AND_HW_CACHE.
    Used for the smooth autocorrelation model and for the fallback scaffold.
    
    Arguments:
        data: datacontainer mapping baselines to complex visibility waterfalls
        weights: datacontainer mapping baselines to real weight waterfalls. 
        filter_delay: maximum delay in ns for the 2D filter
        tslices: dictionary mapping bl to time slices corresponding to low and high bands
        bands: dictionary mapping bl to low band and high band frequency slices
        kwargs: kwargs to pass into sparse_linear_fit_2D()
    
    Returns:
        dpss_fit: datacontainer mapping baselines to 2D DPSS models
    '''
    dpss_fit = copy.deepcopy(data)
    for bl in data.keys():
        # set to all nans by default
        dpss_fit[bl] *= np.nan

        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None) or np.all(weights[bl][tslice, band] == 0):
                continue

            # perform 2D DPSS filter    
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(bl[0:2], tslice, band)]
            time_filters, _ = dpss_operator((data.times[tslice] - data.times[tslice][0]) * 3600 * 24, 
                                            [fr_center], [fr_hw], eigenval_cutoff=[EIGENVAL_CUTOFF])
            freq_filters, _ = dpss_operator(data.freqs[band], [0.0], [filter_delay / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            
            fit, meta = sparse_linear_fit_2D(
                data=data[bl][tslice, band],
                weights=weights[bl][tslice, band],
                axis_1_basis=time_filters,
                axis_2_basis=freq_filters,
                precondition_solver=True,
                iter_lim=CG_ITER_LIM,
                **kwargs,
            )
            dpss_fit[bl][tslice, band] = time_filters.dot(fit).dot(freq_filters.T)
            
    return dpss_fit

In [ ]:
def four_pol_inpainting_figure(ip_data, flags, ip_flags, close=False):
    '''Plots all phase and amplitude waterfalls before and after inpainting for all 4 polarizations in the data.'''
    fig, axes = plt.subplots(4, 4, figsize=(16, 30), sharex=True, sharey=True, dpi=200, gridspec_kw={'wspace': 0.02, 'hspace': 0.01})
    
    vmin = np.nanmin([np.where(~flags[bl] & (np.abs(ip_data[bl]) > 0), np.abs(ip_data[bl]), np.nan) for bl in ip_data])
    vmax = np.nanmax([np.where(~flags[bl] & np.isfinite(ip_data[bl]), np.abs(ip_data[bl]), np.nan) for bl in ip_data])
    lst_grid = ip_data.lsts * 12 / np.pi
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    frac_jds = ip_data.times - int(ip_data.times[0])
    extent = [ip_data.freqs[0] / 1e6, ip_data.freqs[-1] / 1e6, frac_jds[-1], frac_jds[0]]
    
    for row, bl in zip(axes, data):
    
        row[0].imshow(np.where(flags[bl], np.nan, np.angle(ip_data[bl])), aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[1].imshow(np.where(ip_flags[bl], np.nan, np.angle(ip_data[bl])), aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[2].imshow(np.where(flags[bl], np.nan, np.abs(ip_data[bl])), aspect='auto', interpolation='none', norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)
        im = row[3].imshow(np.where(ip_flags[bl], np.nan, np.abs(ip_data[bl])), aspect='auto', interpolation='none', norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)
        
        row[0].set_ylabel(f'JD - {int(ip_data.times[0])}')
        for ax in row:
            ax.tick_params(axis='x', direction='in')
    
        row[0].text(0.02, 0.99, bl, transform=row[0].transAxes, ha='left', va='top', fontsize=12, color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=2))

    # Add LST right axis on rightmost column of last row
    ax2 = axes[-1][-1].twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')
    
    for ax in axes[-1]:
        ax.set_xlabel('Frequency (MHz)')
    
    plt.tight_layout()   
    plt.colorbar(im, ax=axes[0], location='top', label='|V| (Jy)', aspect=50)
    
    if close:
        plt.close(fig)
    return fig

In [ ]:
def subtract_polarized_models(data, flags, model_specs, extra_flags=None, baseline=None):
    """
    Subtract phase-shifted models of polarized point sources from visibility data.

    For each model file, this function reads a model of a polarized source,
    computes the geometric phase shift needed to move the model from the
    phase center to the source's true sky position, and subtracts the
    phased model from the input data in-place. RM/scintillation models are
    zeroed wherever either the model or data are flagged; the foreground
    model intentionally remains applied in flagged samples so it is removed
    from a data-as-scaffold prior.

    Parameters
    ----------
    data : HERAData
        Observed visibility data. Must contain antenna positions (`antpos`),
        frequencies (`freqs`), and observation times (`times`). Modified
        in-place: the model contribution is subtracted from each polarization.
    flags : dict
        Boolean flag dictionary keyed by (ant_i, ant_j, pol) tuples.
        True indicates a flagged (unusable) sample. Used in conjunction
        with model flags to zero out contributions before subtraction.
    model_specs : list of (str, str)
        Pairs of uvh5 model paths and their component names, one per polarized
        source and modeling component.
    baseline : tuple, optional
        The data baseline to process. If omitted, the data must contain exactly
        one baseline.
        Each file must contain:
          - A single baseline's worth of model visibilities
          - `SOURCE_RA` and `SOURCE_DEC` extra keywords (degrees)
            giving the ICRS sky position of the modeled source.

    Raises
    ------
    KeyError
        If a model file is missing the `SOURCE_RA` or `SOURCE_DEC` extra
        keywords, or if a polarization present in the model is absent from
        the data flags dictionary.
    """
    if baseline is None:
        antpairs = list(data.antpairs())
        if len(antpairs) != 1:
            raise ValueError("subtract_polarized_models requires an explicit baseline when data contains multiple baselines.")
        baseline = antpairs[0]

    ai, aj = bl = baseline
    blvec = data.antpos[aj] - data.antpos[ai]  # East-North-Up baseline vector (meters)

    for model_file, model_name in model_specs:
        hd_model = io.HERAData(model_file)
        model_data, model_flags, model_nsamples = hd_model.read()

        model_key = list(model_data.antpairs())[0]  # Baseline key stored in the model file
        model_pols = model_data.pols() 

        # Read the ICRS sky coordinates of the modeled source from file metadata
        right_ascension = hd_model.extra_keywords["SOURCE_RA"]   # degrees
        declination      = hd_model.extra_keywords["SOURCE_DEC"]  # degrees

        # Compute direction cosines (l, m, n) toward the source at each
        # observation time, accounting for Earth rotation and the telescope's
        # geographic location.
        lmn = polfilt.radec_to_lmn(
            right_ascension, declination, data.times, hd_model.telescope.location
        )  # shape: (n_times, 3)

        phasor = np.exp(
            2j * np.pi * np.dot(blvec, lmn)[:, np.newaxis] * hd_model.freqs / constants.c
        )  # shape: (n_times, n_freqs)

        for pol in model_pols:
            # Zero out samples where either the model or the data are flagged,
            # then apply the phase shift to move the model to the source position.
            flags_here = model_flags[model_key + (pol,)] | flags[bl + (pol,)]
            if extra_flags is not None:
                flags_here |= extra_flags

            # Foreground model is relatively high amplitude and will
            # affect the quality of the inpaint if not subtracted in 
            # the gaps
            if model_name == "fg_model":
                bl_model = model_data[model_key + (pol,)] * phasor
            else:
                # Scintillation and RM component are less reliable in gaps
                bl_model = np.where(
                    flags_here,
                    0.0,
                    model_data[model_key + (pol,)] * phasor,
                )  # shape: (n_times, n_freqs)

            # Zero low-band model if requested
            if not USE_LOW_BAND_MODEL:
                bl_model = np.where(
                    data.freqs <= (BAND_SPLIT_FREQ * 1e6),
                    0.0,
                    bl_model,
                )

            # Subtract the phased model from the data in-place
            data[bl + (pol,)] -= bl_model

In [ ]:
scaffold_group_keys = red_groups.RedundantGroups.from_antpos(io.HERAData(single_bl_files[0]).antpos) if len(single_bl_files) > 0 else None
scaffold_dir, scaffold_pattern = os.path.split(INPAINT_SCAFFOLD_FILENAME)
scaffold_files = {}
for scaffold_file in glob.glob(os.path.join(scaffold_dir, scaffold_pattern.format(ant1='*', ant2='*'))):
    match = re.search(scaffold_pattern.replace('.', r'\.').format(ant1=r'(\d+)', ant2=r'(\d+)'), os.path.basename(scaffold_file))
    if match:
        scaffold_files[(int(match.group(1)), int(match.group(2)))] = scaffold_file
print(f'Found {len(scaffold_files)} single-baseline sky models to use as scaffolds in {scaffold_dir}.')


def find_scaffold_file(antpair):
    '''The sky model of the redundant group antpair belongs to, in either orientation, or None.'''
    for ap in scaffold_group_keys.get_red(antpair):
        for candidate in [tuple(ap), tuple(ap)[::-1]]:
            if candidate in scaffold_files:
                return scaffold_files[candidate]
    return None


def load_scaffold(hd, data, scaffold_file):
    '''The sky model matched to hd's times (each to the model integration nearest in LST, within SCAFFOLD_EXTRAP_LIMIT
    model integrations) and rephased to its exact LSTs, keyed like data, with the model's flags. Integrations the model
    does not cover come back fully flagged, so that they count as uncovered and get the fallback scaffold. Returns
    (None, None) if the model lacks a polarization or covers none of the night's integrations.'''
    hdm = io.HERAData(scaffold_file)
    if any(pol not in hdm.pols for pol in data.pols()):
        print(f'\tThe scaffold lacks the {[pol for pol in sorted(data.pols()) if pol not in hdm.pols]} polarization(s).')
        return None, None
    all_model_times, all_model_lsts = abscal.get_all_times_and_lsts(hdm, unwrap=True)
    d2m_time_map = abscal.get_d2m_time_map(data.times, np.unwrap(data.lsts), all_model_times, all_model_lsts,
                                           extrap_limit=SCAFFOLD_EXTRAP_LIMIT)
    matched = np.array([d2m_time_map[time] is not None for time in data.times])
    if not np.any(matched):
        print(f'\tThe scaffold has no integration within {SCAFFOLD_EXTRAP_LIMIT} of its LST spacing of any of the night\'s.')
        return None, None
    if not np.all(matched):
        print(f'\tThe scaffold has no integration within {SCAFFOLD_EXTRAP_LIMIT} of its LST spacing of {np.sum(~matched)} of '
              f'the night\'s {len(matched)} integrations.')
    model_times = np.array([d2m_time_map[time] for time in data.times[matched]])
    model, model_flags, _ = io.partial_time_io(hdm, np.unique(model_times), polarizations=sorted(data.pols()))
    row = {time: i for i, time in enumerate(model.times)}
    rows = np.array([row[time] for time in model_times])
    scaffold, scaffold_flags = {}, {}
    for bl in data:
        scaffold[bl] = np.zeros_like(data[bl])
        scaffold_flags[bl] = np.ones_like(data[bl], dtype=bool)
        scaffold[bl][matched], scaffold_flags[bl][matched] = model[bl][rows], model_flags[bl][rows]
    scaffold = datacontainer.DataContainer(scaffold)
    scaffold.freqs, scaffold.times, scaffold.lsts, scaffold.antpos = data.freqs, data.times, data.lsts, data.antpos
    blvecs = {bl: data.antpos[bl[0]] - data.antpos[bl[1]] for bl in data}
    dlst = np.zeros(len(data.times))
    dlst[matched] = data.lsts[matched] - model.lsts[rows]
    utils.lst_rephase(scaffold, blvecs, data.freqs, dlst, lat=hd.telescope.location.lat.deg, inplace=True)
    return scaffold, datacontainer.DataContainer(scaffold_flags)


def iterative_2D_scaffold(data, flags, weights_before_ip, weights_after_ip, tslices, bands, prefilled=None):
    '''The fallback scaffold (H6C's 2D-informed inpainting): a 2D DPSS model of the data, fit first at ITERATIVE_DELAY_DELTA
    with the gaps zero-weighted, then refit at increasing delays up to INPAINT_DELAY, each time with the gaps filled by the
    previous fit and weighted as if unflagged. prefilled, if given, maps baselines to (values, mask) of flagged cells already
    inpainted against the sky model: they count as data from the first fit on and stay fixed, so that the model informs the
    fit across the edge of its coverage. Returns the final fit (np.nan outside the bands and time slices fit).'''
    if prefilled is None:
        prefilled = {bl: (data[bl], np.zeros_like(flags[bl])) for bl in data}
    current_filter_delay = ITERATIVE_DELAY_DELTA
    dpss_fit = None
    ip_data = None
    while True:
        print(f'\tFitting the fallback scaffold out to {current_filter_delay} ns.')
        if dpss_fit is None:
            # first fit with gaps in data, except where prefilled
            weights = {bl: np.where(prefilled[bl][1], weights_after_ip[bl], weights_before_ip[bl]) for bl in data}
            data_here = copy.deepcopy(data)
            for bl in data_here:
                data_here[bl] = np.where(prefilled[bl][1], prefilled[bl][0], data[bl])
        else:
            # subsequent fits
            weights = weights_after_ip
            data_here = ip_data
        dpss_fit = fit_2D_DPSS(data_here, weights, current_filter_delay, tslices, bands, atol=CG_TOL, btol=CG_TOL)
        ip_data = copy.deepcopy(data)
        for bl in ip_data:
            ip_data[bl] = np.where(prefilled[bl][1], prefilled[bl][0], np.where(flags[bl], dpss_fit[bl], data[bl]))

        # increment current delay until we finally do INPAINT_DELAY
        if current_filter_delay == INPAINT_DELAY:
            break
        current_filter_delay += ITERATIVE_DELAY_DELTA
        if current_filter_delay > INPAINT_DELAY:
            current_filter_delay = INPAINT_DELAY
    return dpss_fit


def feathered_1D_inpaint(data, flags, scaffold, scaffold_flags, weights_after_ip, ip_flags, where_inpainted, tslices, bands,
                         filter_scaffold=False):
    '''H6C's scaffolded, feathered 1D DPSS inpainting: in each integration, the scaffold is substituted where the data are
    flagged, with weights that rise from INPAINT_ZERO_DIST_WEIGHT at a flag edge to full weight INPAINT_WIDTH_FACTOR
    filter-widths away (zero where the scaffold is itself flagged, so the fit interpolates across those cells), and a
    DPSS model out to INPAINT_DELAY is fit along frequency. Returns the inpainted data: the model where where_inpainted,
    the scaffold where flagged but not inpainted, and the data elsewhere.'''
    if filter_scaffold:
        scaff_filter_center = {}
        scaff_filter_half_width = {}
        for bl in data:
            blmag = np.linalg.norm(data.antpos[bl[1]] - data.antpos[bl[0]])
            scaff_filter_center[bl], scaff_filter_half_width[bl] = vis_clean.gen_filter_properties(
                standoff=FILTER_STANDOFF,
                min_dly=FILTER_MIN_DELAY,
                bl_len=blmag / constants.c
            )

    ip_data = copy.deepcopy(data)
    for bl in ip_data:
        # figure out feathering
        distances = np.array([flag_utils.distance_to_nearest_nonzero(~flags[bl][tind, :]) for tind in range(flags[bl].shape[0])])
        width = (1e-9 * INPAINT_DELAY)**-1 / df * INPAINT_WIDTH_FACTOR
        rel_weights = (1 + np.exp(-np.log(INPAINT_ZERO_DIST_WEIGHT**-1 - 1) / width * (distances - width)))**-1

        scaffold_here = np.where(scaffold_flags[bl], 0, scaffold[bl])
        d_mdl = np.full_like(data[bl], np.nan)
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue

            # weights from inpainted autos, except totally-flagged integrations, then multiplied by rel_weights where
            # originally flagged, and zero where the scaffold has nothing to substitute
            wgts = np.where(ip_flags[bl][:, band], 0, weights_after_ip[bl][:, band])
            wgts = np.where(flags[bl][:, band], wgts * rel_weights[:, band], wgts)
            wgts = np.where(flags[bl][:, band] & scaffold_flags[bl][:, band], 0, wgts)
            if np.any(wgts > 0):
                wgts /= np.mean(wgts[wgts > 0])

            # Apply a low-pass delay filter to the scaffold prior to reinpainting
            if filter_scaffold:
                # Only reliable, unflagged scaffold samples should constrain the filter.
                scaffold_good = np.isfinite(scaffold[bl][:, band]) & ~scaffold_flags[bl][:, band]
                sc, _, _ = fourier_filter(data.freqs[band],
                                                  np.where(scaffold_good, scaffold[bl][:, band], 0.0),
                                                  wgts=scaffold_good.astype(float),
                                                  filter_centers=scaff_filter_center[bl],
                                                  filter_half_widths=scaff_filter_half_width[bl],
                                                  mode='dpss_solve',
                                                  eigenval_cutoff=[EIGENVAL_CUTOFF],
                                                  suppression_factors=[EIGENVAL_CUTOFF],
                                                  max_contiguous_edge_flags=len(data.freqs),
                                                  filter_dims=1)
                scaffold_here[:, band] = np.where(scaffold_good, sc, scaffold_here[:, band])

            # 1D DPSS fitting: substitute scaffold where flagged
            d_mdl[:, band], _, _ = fourier_filter(data.freqs[band],
                                                  np.where(flags[bl], scaffold_here, data[bl])[:, band],
                                                  wgts=wgts,
                                                  filter_centers=[0],
                                                  filter_half_widths=[INPAINT_DELAY * 1e-9],
                                                  mode='dpss_solve',
                                                  eigenval_cutoff=[EIGENVAL_CUTOFF],
                                                  suppression_factors=[EIGENVAL_CUTOFF],
                                                  max_contiguous_edge_flags=len(data.freqs),
                                                  filter_dims=1)
        # fill in model where we inpaint, scaffold where flagged but not inpainted, and data otherwise
        ip_data[bl] = np.where(where_inpainted[bl], d_mdl, np.where(flags[bl], scaffold_here, data[bl]))
    return ip_data

## Generate smooth model of autos for noise modeling

In [ ]:
# the night's averaged autocorrelations, for noise modeling
all_outfiles = [outfile for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles]
for outfile in all_outfiles:
    match = re.search(r'\.(\d+)_(\d+)\.', os.path.basename(outfile))
    if match and match.group(1) == match.group(2):
        print(f'Loading {outfile} for autocorrelations to use for noise modeling.')
        hd_autos = io.HERAData(outfile)
        autos, auto_flags, auto_nsamples = hd_autos.read(polarizations=['ee', 'nn'])
        dt = np.median(np.diff(hd_autos.times)) * 24 * 3600
        df = np.median(np.diff(hd_autos.freqs))
        for bl in auto_flags:
            auto_flags[bl] |= prior_flags
        break

In [ ]:
weights = {}
tslices = {}
bands = {}
for bl in autos:
    noise = 2 * np.abs(autos[bl]) / (auto_nsamples[bl] * dt * df)**.5
    weights[bl] = np.where(auto_flags[bl], 0, noise**-2)
    weights[bl] /= np.mean(weights[bl][weights[bl] > 0])
    tslices[bl], bands[bl] = flag_utils.get_minimal_slices(weights[bl] == 0, freqs=autos.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for tslice, band in zip(tslices[bl], bands[bl]):
        cache_fr_center_and_hw(hd_autos, bl[0:2], tslice, band)

In [ ]:
auto_fit = fit_2D_DPSS(autos, weights, AUTO_INPAINT_DELAY, tslices, bands, atol=CG_TOL, btol=CG_TOL)

In [ ]:
# remove unused objects to save memory
del hd_autos, autos, auto_nsamples

## Main loop: scaffolded feathered 1D DPSS inpainting

For each of this job's single-baseline files: apply the final flags, subtract the polarized source
models, find the scaffold, then fit the feathered 1D DPSS model and write the inpainted data, the
filled `nsamples`, and the `where_inpainted` sidecar. When the baseline has a sky model, an
integration counts as *covered* (separately above and below `BAND_SPLIT_FREQ`) if the model has
any unflagged channel at its LST; there the model must be unflagged wherever the data are, or the
notebook raises an error. If the model covers every integration, it is the scaffold. If it covers
only some (an LST range the stack lacks), the covered integrations are inpainted against it first,
then the fallback 2D DPSS scaffold is fit to the whole night with those integrations counted as
gap-free data, so that the model informs the fit across the edge of its coverage, and finally every
integration is inpainted against that 2D fit. Without a sky model, the 2D fit is the scaffold from
the start.

In [ ]:
waterfall_figs = []

for single_bl_file in single_bl_files:

    # load data
    if not os.path.exists(single_bl_file):
        print(f'{single_bl_file} does not exist (likely skipped upstream). Skipping...')
        continue
    print(f"Now loading {single_bl_file}")
    hd = io.HERAData(single_bl_file)
    data, flags, nsamples = hd.read()
    dt = np.median(np.diff(hd.times)) * 24 * 3600
    df = np.median(np.diff(hd.freqs))
    antpair = data.antpairs().pop()
    is_auto = (antpair[0] == antpair[1])

    # apply the night's final flags
    for bl in flags:
        flags[bl] |= prior_flags

    if np.all([flags[bl] for bl in flags]):
        print('\tThis baseline is entirely flagged. Skipping...')
        continue

    if SUBTRACT_POLARIZED_SOURCE and len(model_specs) > 0 and not is_auto:
        print("\tSubtracting off a polarized source model from the visibilities...")
        subtract_polarized_models(data, flags, model_specs, baseline=antpair)

    # get minimal slices, and the fringe-rate ranges the 2D fits use
    tslices = {}
    bands = {}
    for bl in data:
        tslices[bl], bands[bl] = flag_utils.get_minimal_slices(flags[bl], freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
        for tslice, band in zip(tslices[bl], bands[bl]):
            cache_fr_center_and_hw(hd, bl[0:2], tslice, band)

    # fill in effective nsamples at flagged cells (used for weighting, and written out where inpainted)
    ip_nsamples = get_ip_nsamples(nsamples, flags, tslices, bands)

    # get weights from smooth auto model
    weights_before_ip, weights_after_ip = get_weights_for_inpainting(data, flags, tslices, bands, ip_nsamples, auto_fit, auto_flags)

    # set up ip_flags and where_inpainted
    ip_flags = copy.deepcopy(flags)
    where_inpainted = copy.deepcopy(flags)
    for bl in ip_flags:
        ip_flags[bl][:, :] = True
        where_inpainted[bl][:, :] = False
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            ip_flags[bl][tslice, band] = np.all(weights_before_ip[bl][tslice, band] == 0, axis=1, keepdims=True)
            where_inpainted[bl][tslice, band] = flags[bl][tslice, band] & (~ip_flags[bl][tslice, band])

    # the scaffold: the sky model of this redundant group, matched to the night's LSTs, where it has one
    scaffold_file = find_scaffold_file(antpair)
    scaffold = None
    if scaffold_file is not None:
        print(f'\tLoading scaffold from {scaffold_file}')
        scaffold, scaffold_flags = load_scaffold(hd, data, scaffold_file)
    if scaffold is not None and SUBTRACT_POLARIZED_SOURCE and SUBTRACT_POLARIZED_SOURCE_FROM_SCAFFOLD and len(model_specs) > 0 and not is_auto:
        print("\tSubtracting the same polarized source model from the scaffold...")
        subtract_polarized_models(scaffold, scaffold_flags, model_specs, baseline=antpair)

    if scaffold is not None:
        # An integration is covered, separately in each band, where the model has any unflagged cell at that LST; where it
        # is covered, the model must be unflagged wherever the data are (both flagged is fine: the fit interpolates there).
        covered = {bl: np.zeros_like(flags[bl]) for bl in data}
        for bl in data:
            for tslice, band in zip(tslices[bl], bands[bl]):
                if (tslice is None) or (band is None):
                    continue
                covered[bl][:, band] = ~np.all(scaffold_flags[bl][:, band], axis=1, keepdims=True)
                bad = ~flags[bl][:, band] & scaffold_flags[bl][:, band] & covered[bl][:, band]
                if np.any(bad):
                    bad_freqs = data.freqs[band][np.any(bad, axis=0)] / 1e6
                    raise ValueError(f'The scaffold {scaffold_file} is flagged at {np.sum(bad)} cells where {bl} is unflagged, in '
                                     f'{len(bad_freqs)} channels ({bad_freqs.min():.2f}-{bad_freqs.max():.2f} MHz) of '
                                     f'{np.sum(np.any(bad, axis=1))} integrations it otherwise covers.')

        # 1) inpaint every covered integration against the model
        ip_data = feathered_1D_inpaint(data, flags, scaffold, scaffold_flags, weights_after_ip, ip_flags, where_inpainted, tslices, bands,
                                       filter_scaffold=FILTER_SCAFFOLD)
        uncovered = {bl: ~ip_flags[bl] & ~covered[bl] for bl in data}
        n_covered = sum(np.sum(where_inpainted[bl] & covered[bl]) for bl in data)
        n_uncovered = sum(np.sum(where_inpainted[bl] & ~covered[bl]) for bl in data)
        if any(np.any(uncovered[bl]) for bl in data):
            # 2) the model does not cover the whole night: fit the fallback 2D scaffold to all the data, with the
            #    integrations just inpainted counted as gap-free data so that the model informs the fit across its edge...
            print(f'\tThe scaffold covers {n_covered} of the {n_covered + n_uncovered} cells to inpaint: '
                  'fitting a 2D DPSS scaffold to the data, informed by the model where it covers them, for the rest.')
            prefilled = {bl: (ip_data[bl], where_inpainted[bl] & covered[bl]) for bl in data}
            dpss_fit = iterative_2D_scaffold(data, flags, weights_before_ip, weights_after_ip, tslices, bands, prefilled=prefilled)
            # 3) ...and inpaint every integration against it
            ip_data = feathered_1D_inpaint(data, flags, dpss_fit, datacontainer.DataContainer({bl: ~np.isfinite(dpss_fit[bl]) for bl in dpss_fit}),
                                           weights_after_ip, ip_flags, where_inpainted, tslices, bands)
            scaffold_note = f'sky model {scaffold_file} for {n_covered} of {n_covered + n_uncovered} inpainted cells, a 2D DPSS fit of the data informed by it for the rest'
        else:
            scaffold_note = f'sky model {scaffold_file}'
    else:
        print('\tNo sky model for this baseline: building the scaffold from the data with iterative 2D DPSS fits.')
        dpss_fit = iterative_2D_scaffold(data, flags, weights_before_ip, weights_after_ip, tslices, bands)
        ip_data = feathered_1D_inpaint(data, flags, dpss_fit, datacontainer.DataContainer({bl: ~np.isfinite(dpss_fit[bl]) for bl in dpss_fit}),
                                       weights_after_ip, ip_flags, where_inpainted, tslices, bands)
        scaffold_note = 'iterative 2D DPSS fit of the data'
    print(f'\tScaffold: {scaffold_note}.')

    # the filled effective nsamples at the inpainted cells, for LST stacking's weights; other cells keep their own
    ip_nsamples_out = copy.deepcopy(nsamples)
    for bl in ip_nsamples_out:
        ip_nsamples_out[bl] = np.where(where_inpainted[bl], ip_nsamples[bl], nsamples[bl])

    # perform FR=0 filter, if desired
    if FR0_FILTER and not is_auto:
        fr0_filt_ip_data = copy.deepcopy(ip_data)
        for bl in fr0_filt_ip_data:
            for tslice, band in zip(tslices[bl], bands[bl]):
                if (tslice is None) or (band is None):
                    continue
                wgts_here = np.where(ip_flags[bl], 0, weights_after_ip[bl])[tslice, band]
                d_mdl, _, info = fourier_filter(data.times[tslice] * 24 * 60 * 60,
                                                np.where(wgts_here == 0, 0, fr0_filt_ip_data[bl][tslice, band]),
                                                wgts=wgts_here,
                                                filter_centers=[0],
                                                filter_half_widths=[FR0_HALFWIDTH / 1000],
                                                mode='dpss_solve',
                                                eigenval_cutoff=[EIGENVAL_CUTOFF],
                                                suppression_factors=[EIGENVAL_CUTOFF],
                                                max_contiguous_edge_flags=len(data.times),
                                                filter_dims=0)
                fr0_filt_ip_data[bl][tslice, band] -= d_mdl

    # save figures to display later
    if not np.all(list(flags.values())):
        waterfall_figs.append(four_pol_inpainting_figure((fr0_filt_ip_data if (FR0_FILTER and not is_auto) else ip_data),
                                                         flags, ip_flags, close=True))

    # Save inpainting results
    for bl in ip_data:
        if utils.split_bl(bl)[0] == utils.split_bl(bl)[1]:  # is auto
            ip_data[bl][~np.isfinite(ip_data[bl])] = (ip_data[bl][~np.isfinite(ip_data[bl])]).real
    if SAVE_RESULTS:
        assert single_bl_file.endswith(SINGLE_BASELINE_SUFFIX), f'{single_bl_file} does not end with {SINGLE_BASELINE_SUFFIX}.'
        hd.update(data=ip_data, flags=ip_flags, nsamples=ip_nsamples_out)
        hd.history += add_to_history + '\nScaffold: ' + scaffold_note
        outfile = single_bl_file[:-len(SINGLE_BASELINE_SUFFIX)] + INPAINTED_SUFFIX
        print(f"\tNow writing results to {outfile}")
        hd.write_uvh5(outfile, clobber=True)

        # Save fringe-rate filtered results too, if desired (for the auto, just a copy of the inpainted results)
        if FR0_FILTER:
            if not is_auto:
                hd.update(data=fr0_filt_ip_data)
            hd.write_uvh5(outfile.replace('.uvh5', '.FR0_filtered.uvh5'), clobber=True)

        # Save where_inpainted metadata
        hd.update(flags=where_inpainted)
        uvf = UVFlag(hd, mode='flag', copy_flags=True)
        uvf.history += add_to_history
        where_file = single_bl_file[:-len(SINGLE_BASELINE_SUFFIX)] + WHERE_INPAINTED_SUFFIX
        print(f"\tNow writing where-inpainted metadata to {where_file}")
        uvf.write(where_file, clobber=True)

# *Figure 1: 4-Pol Phase and Amplitude Waterfalls Before and After Inpainting*

In [ ]:
for wf_fig in waterfall_figs:
    display(wf_fig)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')